<a href="https://colab.research.google.com/github/jeet8499/pythonproject/blob/main/G1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import threading
import time
import requests
from flask import Flask, request, jsonify

# ==========================================
# 1. MICROSERVICE A: AUTH SERVICE (SQLi)
# ==========================================
auth_app = Flask("AuthService")

# Simple mock database in memory
MOCK_USERS_DB = {
    "admin": "super_secret_password_123",
    "john_doe": "password125"
}

@auth_app.route('/login', methods=['POST'])
def login():
    data = request.get_json() or {}
    username = data.get("username", "")
    password = data.get("password", "")

    # DELIBERATE VULNERABILITY: Raw string concatenation simulating SQL Injection
    # In a real SQLi scenario, an attacker would input: admin' --
    # Here we simulate that logic: if the username contains an override, bypass password check
    if "' OR '1'='1" in username or "' --" in username:
        return jsonify({
            "status": "success",
            "message": "Logged in via SQL Injection bypass!",
            "user": "admin"
        }), 200

    # Standard authentication path
    if username in MOCK_USERS_DB and MOCK_USERS_DB[username] == password:
        return jsonify({"status": "success", "user": username}), 200

    return jsonify({"status": "fail", "message": "Invalid credentials"}), 401


# ==========================================
# 2. MICROSERVICE B: PAYMENT SERVICE (IDOR)
# ==========================================
payment_app = Flask("PaymentService")

# Mock financial data mapped directly to simple integer IDs
MOCK_PAYMENTS = {
    1001: {"user": "john_doe", "amount": "$45.00", "status": "Completed"},
    1002: {"user": "jane_smith", "amount": "$1200.00", "status": "Pending"},
    1003: {"user": "admin", "amount": "$99999.00", "status": "Secret Transfer"}
}

@payment_app.route('/api/payment/<int:payment_id>', methods=['GET'])
def get_payment(payment_id):
    # DELIBERATE VULNERABILITY: IDOR
    # No authentication or authorization check is performed.
    # Any visitor can harvest data by incrementing the payment_id integer.
    payment = MOCK_PAYMENTS.get(payment_id)
    if payment:
        return jsonify({"status": "success", "data": payment}), 200
    return jsonify({"status": "error", "message": "Payment record not found"}), 404


# ==========================================
# 3. BACKGROUND THREAD LAUNCHER
# ==========================================
def run_auth_server():
    # Setting use_reloader=False is mandatory when running Flask in background threads
    auth_app.run(host='127.0.0.1', port=5001, debug=False, use_reloader=False)

def run_payment_server():
    payment_app.run(host='127.0.0.1', port=5002, debug=False, use_reloader=False)

# Safely spawn threads
print("[*] Starting simulated microservices in background threads...")
auth_thread = threading.Thread(target=run_auth_server, daemon=True)
payment_thread = threading.Thread(target=run_payment_server, daemon=True)

auth_thread.start()
payment_thread.start()

# Give the servers a brief moment to boot up safely
time.sleep(2)
print("[✓] Microservices are up and running safely inside Colab memory!")
print("    -> Auth Service running on http://127.0.0.1:5001")
print("    -> Payment Service running on http://127.0.0.1:5002")

[*] Starting simulated microservices in background threads...
 * Serving Flask app 'AuthService'
 * Serving Flask app 'PaymentService'
 * Debug mode: off
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5002
INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5001
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:Press CTRL+C to quit


[✓] Microservices are up and running safely inside Colab memory!
    -> Auth Service running on http://127.0.0.1:5001
    -> Payment Service running on http://127.0.0.1:5002


In [ ]:
print("\n--- Testing Auth Service (SQL Injection Bypass) ---")
sqli_payload = {"username": "admin' --", "password": "wrong_password"}
response_auth = requests.post("http://127.0.0.1:5001/login", json=sqli_payload)
print(f"Status Code: {response_auth.status_code}")
print(f"Response Body: {response_auth.json()}")

print("\n--- Testing Payment Service (IDOR Data Harvesting) ---")
# Accessing record 1003 directly without any session keys or admin flags
response_payment = requests.get("http://127.0.0.1:5002/api/payment/1003")
print(f"Status Code: {response_payment.status_code}")
print(f"Response Body: {response_payment.json()}")

INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:15:05] "POST /login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:15:05] "GET /api/payment/1003 HTTP/1.1" 200 -



--- Testing Auth Service (SQL Injection Bypass) ---
Status Code: 200
Response Body: {'message': 'Logged in via SQL Injection bypass!', 'status': 'success', 'user': 'admin'}

--- Testing Payment Service (IDOR Data Harvesting) ---
Status Code: 200
Response Body: {'data': {'amount': '$99999.00', 'status': 'Secret Transfer', 'user': 'admin'}, 'status': 'success'}


In [ ]:
import requests
import json
from datetime import datetime

class LightweightScanner:
    def __init__(self):
        self.targets = {
            "auth_service": "http://127.0.0.1:5001",
            "payment_service": "http://127.0.0.1:5002"
        }
        self.report = {
            "scan_metadata": {
                "timestamp": datetime.now().isoformat(),
                "status": "Completed"
            },
            "vulnerabilities_found": []
        }

    def check_sql_injection(self):
        """Tests the Auth Service login endpoint using classic SQLi payloads."""
        url = f"{self.targets['auth_service']}/login"
        payloads = [
            {"username": "admin' --", "password": "arbitrary_password"},
            {"username": "' OR '1'='1", "password": "arbitrary_password"}
        ]

        print("[*] Scanning Auth Service for SQL Injection vulnerabilities...")

        for payload in payloads:
            try:
                response = requests.post(url, json=payload, timeout=3)
                # Heuristic: If we get a 200 OK and a success message despite a garbage password,
                # the authentication logic has likely been bypassed.
                if response.status_code == 200 and "success" in response.text.lower():
                    self.report["vulnerabilities_found"].append({
                        "id": "VULN-001",
                        "service": "Auth Service",
                        "endpoint": "/login",
                        "vulnerability_type": "SQL Injection (SQLi)",
                        "severity": "CRITICAL",
                        "description": "The login endpoint concatenates user inputs directly into authentication logic, allowing authentication bypass via classic SQL payloads.",
                        "evidence": f"Payload '{payload['username']}' returned HTTP 200 with successful login signature."
                    })
                    break # One confirmation is enough to flag it
            except requests.exceptions.RequestException as e:
                print(f"[!] Error connecting to Auth Service: {e}")

    def check_idor(self):
        """Tests the Payment Service for IDOR by enumerating sequential object references."""
        print("[*] Scanning Payment Service for Insecure Direct Object Reference (IDOR)...")

        # Test paths sequentially to simulate horizontal authorization harvesting
        base_url = f"{self.targets['payment_service']}/api/payment"
        test_ids = [1001, 1002, 1003]
        accessible_records = []

        try:
            for pid in test_ids:
                url = f"{base_url}/{pid}"
                # Sending a completely unauthenticated request
                response = requests.get(url, timeout=3)

                if response.status_code == 200:
                    accessible_records.append(pid)

            # Heuristic: If multiple independent object records can be fetched
            # with zero auth headers/cookies, an IDOR vulnerability exists.
            if len(accessible_records) > 1:
                self.report["vulnerabilities_found"].append({
                    "id": "VULN-002",
                    "service": "Payment Service",
                    "endpoint": "/api/payment/<payment_id>",
                    "vulnerability_type": "Insecure Direct Object Reference (IDOR)",
                    "severity": "HIGH",
                    "description": "The service leaks private transaction data via direct ID querying without inspecting session ownership, auth tokens, or access control lists.",
                    "evidence": f"Unauthenticated harvesting successful across records: {accessible_records}"
                })
        except requests.exceptions.RequestException as e:
            print(f"[!] Error connecting to Payment Service: {e}")

    def run_scan(self):
        print("=" * 60)
        print("          LAUNCHING LIGHTWEIGHT AGENT SCANNER          ")
        print("=" * 60)

        self.check_sql_injection()
        self.check_idor()

        print("\n[✓] Scan complete. Generating structured JSON artifact...")
        return self.report

# --- EXECUTE THE SCANNER CELL ---
scanner = LightweightScanner()
scan_results = scanner.run_scan()

# Pretty-print the structured JSON output
print("\n=== SYSTEM SECURITY REPORT ARTIFACT ===")
print(json.dumps(scan_results, indent=2))

INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:16:46] "POST /login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:16:46] "GET /api/payment/1001 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:16:46] "GET /api/payment/1002 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:16:46] "GET /api/payment/1003 HTTP/1.1" 200 -


          LAUNCHING LIGHTWEIGHT AGENT SCANNER          
[*] Scanning Auth Service for SQL Injection vulnerabilities...
[*] Scanning Payment Service for Insecure Direct Object Reference (IDOR)...

[✓] Scan complete. Generating structured JSON artifact...

=== SYSTEM SECURITY REPORT ARTIFACT ===
{
  "scan_metadata": {
    "timestamp": "2026-05-23T19:16:46.971645",
    "status": "Completed"
  },
  "vulnerabilities_found": [
    {
      "id": "VULN-001",
      "service": "Auth Service",
      "endpoint": "/login",
      "vulnerability_type": "SQL Injection (SQLi)",
      "severity": "CRITICAL",
      "description": "The login endpoint concatenates user inputs directly into authentication logic, allowing authentication bypass via classic SQL payloads.",
      "evidence": "Payload 'admin' --' returned HTTP 200 with successful login signature."
    },
    {
      "id": "VULN-002",
      "service": "Payment Service",
      "endpoint": "/api/payment/<payment_id>",
      "vulnerability_type": "I

In [ ]:
# Install the modern unified Google GenAI SDK silently
!pip install -q -U google-genai

import json
from google import genai
from google.genai import types
from google.colab import userdata
from IPython.display import Markdown, display

# =====================================================================
# 1. INITIALIZE THE UNIFIED GEMINI CLIENT
# =====================================================================
try:
    # Retrieve the API key securely from Colab's secret manager vault
    api_key = userdata.get('GEMINIAPI')
    client = genai.Client(api_key=api_key)
    print("[✓] Gemini API client initialized successfully using google-genai.")
except Exception as e:
    print(f"[!] Error loading API key. Make sure GEMINI_API_KEY is toggled on in Secrets: {e}")

# =====================================================================
# 2. DEFINE SYSTEM INSTRUCTIONS FOR THE "PRISM" SECURITY TIER
# =====================================================================
prism_system_instruction = """
You are PRISM (Predictive Risk & Incident Security Monitor), an advanced AI core engine
embedded inside an autonomous security agent. Your task is to process raw JSON security reports
and generate execution-ready patching blueprints.

For each vulnerability provided in the scan artifact, you must:
1. Formulate a risk assessment detailing the operational blast radius.
2. Provide precise, actionable Python code or structural modifications to patch the vulnerability.
3. Keep the output highly concise, objective, and structured. Do not use filler introductory words.
"""

# =====================================================================
# 3. CONSTRUCT THE PROMPT WINDOW & EXECUTE INFERENCE
# =====================================================================
# We use 'scan_results' which was stored in the memory from our previous cell run
scan_payload_str = json.dumps(scan_results, indent=2)
# =====================================================================
# 3. CONSTRUCT THE PROMPT WINDOW & EXECUTE INFERENCE
# =====================================================================
# Clean, explicit multiline formatting with zero hidden trailing characters
prompt_content = (
    f"Analyze the following vulnerability scan data collected from our simulated microservices.\n"
    f"Generate a clear remediation plan for each flagged item.\n\n"
    f"VULNERABILITY ARTIFACT:\n"
    f"```json\n{json.dumps(scan_results, indent=2)}\n```"
)

[✓] Gemini API client initialized successfully using google-genai.


In [ ]:
import sys
from flask import request, jsonify

# =====================================================================
# 1. DEFINE SECURE PATCHES WITH CORRECT VIEW MATCHING NAMES
# =====================================================================

def login():
    """Secure replacement function matching original view name exactly."""
    data = request.get_json() or {}
    username = data.get("username", "")
    password = data.get("password", "")

    # Strict key-value checking removes injection entrypoints completely
    if username in MOCK_USERS_DB and MOCK_USERS_DB[username] == password:
        return jsonify({"status": "success", "user": username}), 200

    return jsonify({"status": "fail", "message": "Invalid credentials"}), 401


def get_payment(payment_id):
    """Secure replacement function matching original view name exactly."""
    # Strict Authorization Access Check
    current_session_user = request.headers.get("X-Authenticated-User")

    payment = MOCK_PAYMENTS.get(payment_id)
    if not payment:
        return jsonify({"status": "error", "message": "Payment record not found"}), 404

    # FIX: Explicit identity verification check block
    if payment["user"] != current_session_user and current_session_user != "admin":
        return jsonify({
            "status": "error",
            "message": "Access Denied: Unauthorized to view this billing statement."
        }), 403

    return jsonify({"status": "success", "data": payment}), 200


# =====================================================================
# 2. BULLETPROOF INCIDENTFORGE ORCHESTRATION ENGINE
# =====================================================================
class IncidentForgeEngine:
    @staticmethod
    def deploy_hotpatch(service_name, target_view_name, secure_function):
        """
        Directly targets and overwrites the exact view function binding inside memory
        using the framework-level execution context.
        """
        print(f"[*] IncidentForge: Initiating live hot-swap targeting view: [{target_view_name}]...")

        if service_name == "Auth Service":
            target_app = auth_app
        elif service_name == "Payment Service":
            target_app = payment_app
        else:
            print(f"[!] Unknown service: {service_name}")
            return False

        if target_view_name in target_app.view_functions:
            # Atomic swap of the functional reference in runtime memory
            target_app.view_functions[target_view_name] = secure_function
            print(f"[✓] IncidentForge: Hot-swap successful for [{service_name}] -> {target_view_name}")
            return True
        else:
            print(f"[!] Error: Target view function '{target_view_name}' not registered in Flask table.")
            return False

# =====================================================================
# 3. EXECUTE REMEDIATION
# =====================================================================
print("=" * 60)
print("             DEPLOYNIG STABLE INCIDENTFORGE FIX               ")
print("=" * 60)

# Hot-swap Auth Service
IncidentForgeEngine.deploy_hotpatch(
    service_name="Auth Service",
    target_view_name="login",
    secure_function=login
)

# Hot-swap Payment Service using matching function endpoint signature name
print("")
IncidentForgeEngine.deploy_hotpatch(
    service_name="Payment Service",
    target_view_name="get_payment",
    secure_function=get_payment
)

print("\n[✓] Live core functions mutated. Run validation cell to check protection state.")

             DEPLOYNIG STABLE INCIDENTFORGE FIX               
[*] IncidentForge: Initiating live hot-swap targeting view: [login]...
[✓] IncidentForge: Hot-swap successful for [Auth Service] -> login

[*] IncidentForge: Initiating live hot-swap targeting view: [get_payment]...
[✓] IncidentForge: Hot-swap successful for [Payment Service] -> get_payment

[✓] Live core functions mutated. Run validation cell to check protection state.


In [ ]:
print("\n--- [Validation Loop] Retesting Auth Service with Malicious Payload ---")
sqli_payload = {"username": "admin' --", "password": "wrong_password"}
res_auth = requests.post("http://127.0.0.1:5001/login", json=sqli_payload)
print(f"Status Code: {res_auth.status_code} (Expected: 401 Unauthorized)")
print(f"Response: {res_auth.json()}")

print("\n--- [Validation Loop] Retesting Payment Service for IDOR Exploit ---")
# Attempting to scan record 1003 without sending an authorized identification header
res_payment = requests.get("http://127.0.0.1:5002/api/payment/1003")
print(f"Status Code: {res_payment.status_code} (Expected: 403 Forbidden or 404/Error due to lack of header)")
print(f"Response: {res_payment.json()}")

INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:32:54] "POST /login HTTP/1.1" 401 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:32:54] "GET /api/payment/1003 HTTP/1.1" 403 -



--- [Validation Loop] Retesting Auth Service with Malicious Payload ---
Status Code: 401 (Expected: 401 Unauthorized)
Response: {'message': 'Invalid credentials', 'status': 'fail'}

--- [Validation Loop] Retesting Payment Service for IDOR Exploit ---
Status Code: 403 (Expected: 403 Forbidden or 404/Error due to lack of header)
Response: {'message': 'Access Denied: Unauthorized to view this billing statement.', 'status': 'error'}


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import requests
import json

# Global metric counters tracking the active session state
metrics = {
    "auth_blocks": 0,
    "payment_blocks": 0,
    "total_scans": 1
}

# =====================================================================
# 1. DEFINE SIMULATION ACTIONS (Hits your running background threads)
# =====================================================================
def trigger_sqli_attack():
    url = "http://127.0.0.1:5001/login"
    payload = {"username": "admin' --", "password": "wrong_password"}
    try:
        res = requests.post(url, json=payload, timeout=2)
        metrics["auth_blocks"] += 1
        return res.status_code, res.json()
    except Exception as e:
        return "ERR", str(e)

def trigger_idor_attack():
    url = "http://127.0.0.1:5002/api/payment/1003"
    try:
        res = requests.get(url, timeout=2)
        metrics["payment_blocks"] += 1
        return res.status_code, res.json()
    except Exception as e:
        return "ERR", str(e)

# =====================================================================
# 2. RENDER THE INTERACTIVE HTML/CSS INTERFACE
# =====================================================================
output_area = widgets.Output()

def render_dashboard(log_msg="System initialized. Monitoring live channels..."):
    with output_area:
        clear_output(wait=True)

        # UI Styling Blueprint (Clean, modern, and dark-mode compatible)
        html_layout = f"""
        <div style="font-family: sans-serif; background-color: #121214; color: #e4e4e7; padding: 20px; border-radius: 12px; border: 1px solid #27272a;">
            <!-- Header Section -->
            <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #27272a; padding-bottom: 12px; margin-bottom: 15px;">
                <h2 style="color: #38bdf8; margin: 0; font-weight: 600;">PRISM × IncidentForge Control Plane</h2>
                <span style="background-color: #22c55e20; color: #22c55e; border: 1px solid #22c55e40; padding: 4px 10px; border-radius: 20px; font-size: 12px; font-weight: bold;">● AGENT ACTIVE</span>
            </div>

            <!-- Metric Performance Cards Grid -->
            <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 15px; margin-bottom: 20px;">
                <div style="background-color: #1c1c1f; padding: 15px; border-radius: 8px; border: 1px solid #27272a; text-align: center;">
                    <div style="font-size: 12px; color: #a1a1aa; text-transform: uppercase;">SQLi Blocks</div>
                    <div style="font-size: 28px; font-weight: bold; color: #f43f5e; margin-top: 5px;">{metrics['auth_blocks']}</div>
                </div>
                <div style="background-color: #1c1c1f; padding: 15px; border-radius: 8px; border: 1px solid #27272a; text-align: center;">
                    <div style="font-size: 12px; color: #a1a1aa; text-transform: uppercase;">IDOR Blocks</div>
                    <div style="font-size: 28px; font-weight: bold; color: #fbbf24; margin-top: 5px;">{metrics['payment_blocks']}</div>
                </div>
                <div style="background-color: #1c1c1f; padding: 15px; border-radius: 8px; border: 1px solid #27272a; text-align: center;">
                    <div style="font-size: 12px; color: #a1a1aa; text-transform: uppercase;">System Health</div>
                    <div style="font-size: 28px; font-weight: bold; color: #22c55e; margin-top: 5px;">100%</div>
                </div>
            </div>

            <!-- Deployment Status Targets -->
            <h3 style="color: #e4e4e7; margin-bottom: 10px; font-size: 14px;">Managed Microservices Pool</h3>
            <div style="margin-bottom: 20px;">
                <div style="display: flex; justify-content: space-between; background-color: #1c1c1f; padding: 10px 15px; margin-bottom: 8px; border-radius: 6px; border-left: 4px solid #22c55e;">
                    <span><strong>Auth Service</strong> (Port 5001)</span>
                    <span style="color: #22c55e; font-weight: bold;">[✓] Hot-Patched & Protected</span>
                </div>
                <div style="display: flex; justify-content: space-between; background-color: #1c1c1f; padding: 10px 15px; border-radius: 6px; border-left: 4px solid #22c55e;">
                    <span><strong>Payment Service</strong> (Port 5002)</span>
                    <span style="color: #22c55e; font-weight: bold;">[✓] Hot-Patched & Protected</span>
                </div>
            </div>

            <!-- Terminal Incident Log Stream -->
            <h3 style="color: #e4e4e7; margin-bottom: 10px; font-size: 14px;">Live Incident / Agent Signal Log</h3>
            <div style="background-color: #09090b; font-family: monospace; padding: 12px; border-radius: 6px; border: 1px solid #27272a; font-size: 13px; color: #34d399; min-height: 60px; max-height: 120px; overflow-y: auto;">
                {log_msg}
            </div>
        </div>
        """
        display(HTML(html_layout))

# =====================================================================
# 3. INTERACTIVE CONTROL WIDGET WIREUP
# =====================================================================
btn_sqli = widgets.Button(description="⚠️ Fire SQLi Exploit", button_style='danger', layout=widgets.Layout(width='48%', margin='5px'))
btn_idor = widgets.Button(description="🔒 Fire IDOR Exploit", button_style='warning', layout=widgets.Layout(width='48%', margin='5px'))
btn_reset = widgets.Button(description="🔄 Reset Telemetry Metrics", button_style='info', layout=widgets.Layout(width='98%', margin='10px 5px'))

def on_sqli_click(b):
    status, body = trigger_sqli_attack()
    log = f"[ATTACK REJECTED] Auth Service intercepted SQLi payload. HTTP {status} Sent -> Response: {json.dumps(body)}"
    render_dashboard(log)

def on_idor_click(b):
    status, body = trigger_idor_attack()
    log = f"[ATTACK REJECTED] Payment Service blocked illicit parameter mapping. HTTP {status} Sent -> Response: {json.dumps(body)}"
    render_dashboard(log)

def on_reset_click(b):
    metrics["auth_blocks"] = 0
    metrics["payment_blocks"] = 0
    render_dashboard("Telemetry system counter reset clean. Awaiting live payloads...")

# Register button trigger callbacks
btn_sqli.on_click(on_sqli_click)
btn_idor.on_click(on_idor_click)
btn_reset.on_click(on_reset_click)

# Display the dashboard components visually stacked inside your notebook output
control_row = widgets.HBox([btn_sqli, btn_idor])
ui_container = widgets.VBox([output_area, control_row, btn_reset])

display(ui_container)
render_dashboard() # Render the default container state instantly

INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:35:28] "POST /login HTTP/1.1" 401 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:35:35] "GET /api/payment/1003 HTTP/1.1" 403 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:35:44] "POST /login HTTP/1.1" 401 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:35:46] "POST /login HTTP/1.1" 401 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:35:46] "POST /login HTTP/1.1" 401 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:35:48] "GET /api/payment/1003 HTTP/1.1" 403 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:50:12] "POST /login HTTP/1.1" 401 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:50:20] "POST /login HTTP/1.1" 401 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:50:21] "POST /login HTTP/1.1" 401 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:50:25] "POST /login HTTP/1.1" 401 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:50:34] "GET /api/payment/1003 HTTP/1.1" 403 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 19:50:40] "GET /api/payment/1003 HTTP/1.1" 403 -
